In [1]:
from google.colab import drive
drive.mount('/content/drive/')

Mounted at /content/drive/


In [2]:
cd /content/drive/MyDrive/project/

/content/drive/.shortcut-targets-by-id/1IhtrClOvYkKUSZxwUoLOhxwHY9CdXCaH/project


In [3]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import transforms
import sys
import os

# --- PATH SETUP ---
PROJECT_ROOT = os.getcwd()

In [4]:
PROJECT_ROOT

'/content/drive/.shortcut-targets-by-id/1IhtrClOvYkKUSZxwUoLOhxwHY9CdXCaH/project'

In [5]:
dataset_path = PROJECT_ROOT+"/tcdata"
images_path = dataset_path+"/images"
label_path = dataset_path+"/labels"

In [6]:
train_data_zip = dataset_path+"/mchar_train.zip"
val_data_zip = dataset_path+"/mchar_val.zip"
test_data_zip = dataset_path+"/mchar_test_a.zip"

In [7]:
import zipfile
import os

In [9]:
!unzip -u ./tcdata/mchar_test_a.zip -d ./tcdata/images/

Se han truncado las últimas 5000 líneas del flujo de salida.
  inflating: ./tcdata/images/__MACOSX/mchar_test_a/._019196.png  
  inflating: ./tcdata/images/mchar_test_a/039246.png  
  inflating: ./tcdata/images/__MACOSX/mchar_test_a/._039246.png  
  inflating: ./tcdata/images/mchar_test_a/021409.png  
  inflating: ./tcdata/images/__MACOSX/mchar_test_a/._021409.png  
  inflating: ./tcdata/images/mchar_test_a/018288.png  
  inflating: ./tcdata/images/__MACOSX/mchar_test_a/._018288.png  
  inflating: ./tcdata/images/mchar_test_a/020717.png  
  inflating: ./tcdata/images/__MACOSX/mchar_test_a/._020717.png  
  inflating: ./tcdata/images/mchar_test_a/038158.png  
  inflating: ./tcdata/images/__MACOSX/mchar_test_a/._038158.png  
  inflating: ./tcdata/images/mchar_test_a/001901.png  
  inflating: ./tcdata/images/__MACOSX/mchar_test_a/._001901.png  
  inflating: ./tcdata/images/mchar_test_a/027078.png  
  inflating: ./tcdata/images/__MACOSX/mchar_test_a/._027078.png  
  inflating: ./tcdata/imag

In [ ]:
!unzip -u ./tcdata/mchar_train.zip -d ./tcdata/images/
!unzip -u ./tcdata/mchar_val.zip -d ./tcdata/images/

Se han truncado las últimas 5000 líneas del flujo de salida.
  inflating: ./tcdata/images/mchar_train/012400.png  
  inflating: ./tcdata/images/mchar_train/002613.png  
  inflating: ./tcdata/images/mchar_train/014071.png  
  inflating: ./tcdata/images/mchar_train/004262.png  
  inflating: ./tcdata/images/mchar_train/004504.png  
  inflating: ./tcdata/images/mchar_train/013078.png  
  inflating: ./tcdata/images/mchar_train/014717.png  
  inflating: ./tcdata/images/mchar_train/015409.png  
  inflating: ./tcdata/images/mchar_train/002175.png  
  inflating: ./tcdata/images/mchar_train/012366.png  
  inflating: ./tcdata/images/mchar_train/009020.png  
  inflating: ./tcdata/images/mchar_train/006313.png  
  inflating: ./tcdata/images/mchar_train/019233.png  
  inflating: ./tcdata/images/mchar_train/016100.png  
  inflating: ./tcdata/images/mchar_train/000762.png  
  inflating: ./tcdata/images/mchar_train/010571.png  
  inflating: ./tcdata/images/mchar_train/021974.png  
  inflating: ./tcdata

In [ ]:
!unzip -u ./tcdata/mchar_val.zip -d ./tcdata/images/

Se han truncado las últimas 5000 líneas del flujo de salida.
  inflating: ./tcdata/images/__MACOSX/mchar_val/._004842.png  
  inflating: ./tcdata/images/mchar_val/005584.png  
  inflating: ./tcdata/images/__MACOSX/mchar_val/._005584.png  
  inflating: ./tcdata/images/mchar_val/007393.png  
  inflating: ./tcdata/images/__MACOSX/mchar_val/._007393.png  
  inflating: ./tcdata/images/mchar_val/001084.png  
  inflating: ./tcdata/images/__MACOSX/mchar_val/._001084.png  
  inflating: ./tcdata/images/mchar_val/006933.png  
  inflating: ./tcdata/images/__MACOSX/mchar_val/._006933.png  
  inflating: ./tcdata/images/mchar_val/003693.png  
  inflating: ./tcdata/images/__MACOSX/mchar_val/._003693.png  
  inflating: ./tcdata/images/mchar_val/003687.png  
  inflating: ./tcdata/images/__MACOSX/mchar_val/._003687.png  
  inflating: ./tcdata/images/mchar_val/002599.png  
  inflating: ./tcdata/images/__MACOSX/mchar_val/._002599.png  
  inflating: ./tcdata/images/mchar_val/001090.png  
  inflating: ./tcda

In [ ]:
def extract_zip(path, file):
  # Extract train.zip
  with zipfile.ZipFile(file, 'r') as zip_ref:
      zip_ref.extractall(path)
      print(f"✓ {file} extracted in {path}")

extract_zip(images_path, train_data_zip)
extract_zip(images_path, val_data_zip)
extract_zip(images_path, test_data_zip)

KeyboardInterrupt: 

In [ ]:
train_data_path = images_path + "/mchar_train"
val_data_path = images_path + "/mchar_val"
test_data_path = images_path +"/mchar_test_a"

In [ ]:
# Extract labels and convert them to YOLO format
import json
import cv2
import os

def compute_yolo_coordinates(top, left, bottom, right, img_width, img_height):
    """
    Transforms absolute coordinates to YOLO format

    Args:
      top: Y coordinate of the top-left corner
      left: X coordinate of the top-left corner
      bottom: Y coordinate of the bottom-right corner
      right: X coordinate of the bottom-right corner
      img_width: image width
      img_height: image height

    Returns:
      x_center, y_center, width, height (normalized)
    """
    # Calculate center
    x_center = (left + right) / 2.0 / img_width
    y_center = (top + bottom) / 2.0 / img_height

    # Calculate width and height
    width = (right - left) / img_width
    height = (bottom - top) / img_height

    return x_center, y_center, width, height

def toYoloFormat(json_file, json_path, img_path):
    """
    Converts JSON annotations to YOLO format

    Args:
      json_file: path to the JSON annotation file
      json_path: directory to save the .txt files
      img_path: directory where images are located
    """
    with open(json_file, "r") as f:
        meta = json.load(f)

    keys = list(meta.keys()) # get keys to access

    # Create output directory if it doesn't exist
    os.makedirs(json_path, exist_ok=True)

    for key in keys:
        tops = meta[key]['top']
        heights = meta[key]['height']
        lefts = meta[key]['left']
        widths = meta[key]['width']
        labels = meta[key]['label']

        # Read the corresponding image
        img_full_path = os.path.join(img_path, key)

        # Verify the image exists
        if not os.path.exists(img_full_path):
            print(f"Warning: Image {img_full_path} not found")
            continue

        img = cv2.imread(img_full_path)

        if img is None:
            print(f"Warning: Could not read image {img_full_path}")
            continue

        shape = img.shape
        img_h = shape[0]  # height
        img_w = shape[1]  # width

        # The filename will be the same as the image but with .txt extension
        name_file = key.replace(".png", ".txt")
        output_path = os.path.join(json_path, name_file)

        with open(output_path, 'w') as f:

            for i in range(len(labels)):
                # Calculate YOLO coordinates
                # Note: bottom = top + height, right = left + width
                x_center, y_center, width_norm, height_norm = compute_yolo_coordinates(
                    tops[i],
                    lefts[i],
                    tops[i] + heights[i],  # bottom
                    lefts[i] + widths[i],  # right
                    img_w, img_h
                )

                # Write in YOLO format: class x_center y_center width height
                f.write(f"{labels[i]} {x_center:.6f} {y_center:.6f} {width_norm:.6f} {height_norm:.6f}\n")

    print(f"Conversion completed. Files saved in: {json_path}")

In [ ]:
toYoloFormat(dataset_path+"/mchar_train.json", label_path+"/mchar_train", images_path+"/mchar_train")


KeyboardInterrupt: 

In [ ]:
toYoloFormat(dataset_path+"/mchar_val.json", label_path+"/mchar_val", images_path+"/mchar_val")

Conversion completed. Files saved in: /content/drive/.shortcut-targets-by-id/1IhtrClOvYkKUSZxwUoLOhxwHY9CdXCaH/project/tcdata/labels/mchar_val
